# Zero-3DCE Colab Implementation with Google Drive Checkpoints

This version is adjusted for the thesis Google Drive workflow. It mounts Google Drive, extracts training/testing frames from Drive videos, trains Zero-3DCE, saves selected epoch checkpoints, and saves the best checkpoint based on the lowest training loss.

Default Drive layout used in this notebook:

```text
/content/drive/MyDrive/Thesis2/Thesis2_6fpsRawVideos/
  midnight_6fps.mp4
  evening_6fps.mp4

/content/drive/MyDrive/Thesis2/zero3dce_colab_outputs/
  outputs_extracted_frames/
  outputs_enhanced_frames/
  outputs_enhanced_videos/
  saved_runs/
```

Change only the path variables in the Google Drive paths cell if your folder names are different.


In [3]:

# ===============================
# 1) COLAB SETUP
# ===============================
!pip -q install pytorch-msssim opencv-python tqdm

import os, glob, math, random, shutil
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid

try:
    from pytorch_msssim import ms_ssim
    HAS_MSSSIM = True
except Exception:
    HAS_MSSSIM = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Upload Zero-DCE-master.zip to Colab first, or place it at /content/Zero-DCE-master.zip.
# This is optional; the Zero-3DCE code below is self-contained, but unzipping helps compare with the original repo.
if Path('/content/Zero-DCE-master.zip').exists():
    !unzip -q -o /content/Zero-DCE-master.zip -d /content/
    print('Zero-DCE zip extracted.')
else:
    print('No /content/Zero-DCE-master.zip found. Continuing with self-contained Zero-3DCE code.')


Device: cuda
No /content/Zero-DCE-master.zip found. Continuing with self-contained Zero-3DCE code.


In [4]:

# ===============================
# 1B) GOOGLE DRIVE PATHS
# ===============================
from google.colab import drive
drive.mount('/content/drive')

# Main thesis folders
RAW_VIDEO_DIR = Path('/content/drive/MyDrive/Thesis2/Thesis2_6fpsRawVideos')

OUTPUT_ROOT = Path('/content/drive/MyDrive/Thesis2/outputs_zero3dce_v5sofia')

# Training/testing videos
TRAIN_VIDEO_NAME = 'midnight_6fps.mp4'
TEST_VIDEO_NAME = 'evening_6fps.mp4'

# Output folders in Drive
TRAIN_FRAMES_DIR = OUTPUT_ROOT / 'outputs_extracted_frames' / 'midnight_train_frames'
TEST_FRAMES_DIR = OUTPUT_ROOT / 'outputs_extracted_frames' / 'evening_test_frames'
ENHANCED_FRAMES_DIR = OUTPUT_ROOT / 'outputs_enhanced_frames' / 'evening_enhanced_frames'
ENHANCED_VIDEO_PATH = OUTPUT_ROOT / 'outputs_enhanced_videos' / 'evening_zero3dce_enhanced.mp4'
SAVED_RUNS_DIR = OUTPUT_ROOT / 'saved_runs'

for folder in [TRAIN_FRAMES_DIR, TEST_FRAMES_DIR, ENHANCED_FRAMES_DIR, ENHANCED_VIDEO_PATH.parent, SAVED_RUNS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Train video:', RAW_VIDEO_DIR / TRAIN_VIDEO_NAME)
print('Test video :', RAW_VIDEO_DIR / TEST_VIDEO_NAME)
print('Save dir   :', SAVED_RUNS_DIR)
print('Train video found:', (RAW_VIDEO_DIR / TRAIN_VIDEO_NAME).exists())
print('Test video found :', (RAW_VIDEO_DIR / TEST_VIDEO_NAME).exists())


Mounted at /content/drive
Train video: /content/drive/MyDrive/Thesis2/Thesis2_6fpsRawVideos/midnight_6fps.mp4
Test video : /content/drive/MyDrive/Thesis2/Thesis2_6fpsRawVideos/evening_6fps.mp4
Save dir   : /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs
Train video found: True
Test video found : True


In [5]:

# ===============================
# 2) OPTIONAL: EXTRACT VIDEO FRAMES
# ===============================
def extract_video_frames(video_path, out_dir, target_fps=6, max_frames=None):
    """Extract frames from a video into out_dir at approximately target_fps."""
    video_path = str(video_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    step = max(int(round(src_fps / target_fps)), 1)

    saved, idx = 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            Image.fromarray(frame).save(out_dir / f"frame_{saved:06d}.jpg")
            saved += 1
            if max_frames and saved >= max_frames:
                break
        idx += 1
    cap.release()
    print(f"Saved {saved} frames to {out_dir}")

# Example:
# extract_video_frames('/content/my_low_light_video.mp4', '/content/data/train/video_001', target_fps=6)


In [6]:

# ===============================
# 2B) EXTRACT THESIS VIDEO FRAMES TO DRIVE
# ===============================
# This runs only if the target frame folder is still empty.
# Delete the frame folder or set FORCE_REEXTRACT=True if you want to extract again.
FORCE_REEXTRACT = False

if FORCE_REEXTRACT and TRAIN_FRAMES_DIR.exists():
    shutil.rmtree(TRAIN_FRAMES_DIR)
    TRAIN_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

if not any(TRAIN_FRAMES_DIR.glob('*.jpg')):
    extract_video_frames(RAW_VIDEO_DIR / TRAIN_VIDEO_NAME, TRAIN_FRAMES_DIR, target_fps=6)
else:
    print('Training frames already exist:', TRAIN_FRAMES_DIR)

if FORCE_REEXTRACT and TEST_FRAMES_DIR.exists():
    shutil.rmtree(TEST_FRAMES_DIR)
    TEST_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

if not any(TEST_FRAMES_DIR.glob('*.jpg')):
    extract_video_frames(RAW_VIDEO_DIR / TEST_VIDEO_NAME, TEST_FRAMES_DIR, target_fps=6)
else:
    print('Testing frames already exist:', TEST_FRAMES_DIR)


Saved 12242 frames to /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/outputs_extracted_frames/midnight_train_frames
Saved 12242 frames to /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/outputs_extracted_frames/evening_test_frames


In [7]:

# ===============================
# 3) DATASET: IMAGE OR VIDEO CLIPS
# ===============================
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

class LowLightClipDataset(Dataset):
    """
    Returns clips as tensors shaped [C, D, H, W], normalized to [0, 1].
    - If train_root contains subfolders, each subfolder is treated as a video sequence.
    - If train_root contains loose images, each image is duplicated into a clip.
    """
    def __init__(self, train_root, clip_len=2, image_size=256, stride=1):
        self.train_root = Path(train_root)
        self.clip_len = clip_len
        self.image_size = image_size
        self.stride = stride
        self.samples = []

        self.tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

        subdirs = [p for p in self.train_root.iterdir() if p.is_dir()] if self.train_root.exists() else []
        for folder in subdirs:
            frames = sorted([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS])
            if len(frames) >= clip_len:
                for i in range(0, len(frames) - clip_len + 1, stride):
                    self.samples.append(frames[i:i+clip_len])
            elif len(frames) > 0:
                self.samples.append([frames[0]] * clip_len)

        loose = sorted([p for p in self.train_root.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]) if self.train_root.exists() else []
        for img in loose:
            self.samples.append([img] * clip_len)

        if len(self.samples) == 0:
            raise RuntimeError(f"No images found in {train_root}. Put frames in subfolders or loose images in the train folder.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        paths = self.samples[idx]
        frames = []
        for p in paths:
            img = Image.open(p).convert('RGB')
            frames.append(self.tf(img))  # [C,H,W]
        clip = torch.stack(frames, dim=1)  # [C,D,H,W]
        return clip

# Quick check:
# ds = LowLightClipDataset('/content/data/train', clip_len=2, image_size=256)
# print(len(ds), ds[0].shape)  # [3, D, 256, 256]


In [8]:

# ===============================
# 4) ZERO-3DCE MODEL
# ===============================
class DepthwiseSeparableConv3D(nn.Module):
    """Depthwise 3D convolution + pointwise 1x1x1 convolution."""
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch, bias=True)
        self.pointwise = nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=True)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

class SpatialAttention3D(nn.Module):
    """3D spatial attention using channel-wise average and max projections."""
    def __init__(self, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv3d(2, 1, kernel_size=kernel_size, padding=pad, bias=False)

    def forward(self, x):
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.sigmoid(self.conv(torch.cat([avg_map, max_map], dim=1)))
        return x * attn

class Zero3DCE(nn.Module):
    """
    Practical Zero-3DCE network.
    Input : [B, 3, D, H, W]
    Output: enhanced clip [B, 3, D, H, W], intermediate enhanced clip, and 24 curve maps.
    """
    def __init__(self, nf=32):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)

        self.conv1 = DepthwiseSeparableConv3D(3, nf)
        self.conv2 = DepthwiseSeparableConv3D(nf, nf)
        self.conv3 = DepthwiseSeparableConv3D(nf, nf)
        self.conv4 = DepthwiseSeparableConv3D(nf, nf)
        self.conv5 = DepthwiseSeparableConv3D(nf * 2, nf)
        self.conv6 = DepthwiseSeparableConv3D(nf * 2, nf)
        self.conv7 = DepthwiseSeparableConv3D(nf * 2, 24)

        self.att1 = SpatialAttention3D(7)
        self.att2 = SpatialAttention3D(7)
        self.att3 = SpatialAttention3D(7)
        self.pool = nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))

    def _up_to(self, x, ref):
        return F.interpolate(x, size=ref.shape[2:], mode='trilinear', align_corners=False)

    def forward(self, x):
        x1 = self.att1(self.relu(self.conv1(x)))
        p1 = self.pool(x1)
        x2 = self.att2(self.relu(self.conv2(p1)))
        p2 = self.pool(x2)
        x3 = self.att3(self.relu(self.conv3(p2)))
        p3 = self.pool(x3)
        x4 = self.relu(self.conv4(p3))

        u4 = self._up_to(x4, x3)
        x5 = self.relu(self.conv5(torch.cat([x3, u4], dim=1)))
        u5 = self._up_to(x5, x2)
        x6 = self.relu(self.conv6(torch.cat([x2, u5], dim=1)))
        u6 = self._up_to(x6, x1)
        r = torch.tanh(self.conv7(torch.cat([x1, u6], dim=1)))

        curves = torch.split(r, 3, dim=1)
        y = x
        enhanced_1 = None
        for i, a in enumerate(curves):
            y = y + a * (torch.pow(y, 2) - y)
            y = torch.clamp(y, 0.0, 1.0)
            if i == 3:
                enhanced_1 = y
        return enhanced_1, y, r

def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Conv3d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


In [9]:

# ===============================
# 5) ZERO-REFERENCE LOSSES FOR 3D CLIPS
# ===============================
class ColorConstancyLoss3D(nn.Module):
    def forward(self, x):
        mean_rgb = x.mean(dim=(3, 4), keepdim=True)
        mr, mg, mb = torch.split(mean_rgb, 1, dim=1)
        loss = torch.sqrt((mr - mg).pow(2) + (mr - mb).pow(2) + (mb - mg).pow(2) + 1e-6)
        return loss.mean()

class ExposureLoss3D(nn.Module):
    def __init__(self, patch_size=16, target=0.6):
        super().__init__()
        self.pool = nn.AvgPool3d(kernel_size=(1, patch_size, patch_size), stride=(1, patch_size, patch_size))
        self.target = target
    def forward(self, x):
        gray = x.mean(dim=1, keepdim=True)
        patch_mean = self.pool(gray)
        return torch.mean((patch_mean - self.target).pow(2))

class SpatialConsistencyLoss3D(nn.Module):
    def __init__(self, patch_size=4):
        super().__init__()
        self.pool = nn.AvgPool3d(kernel_size=(1, patch_size, patch_size), stride=(1, patch_size, patch_size))
    def forward(self, org, enh):
        org = self.pool(org.mean(dim=1, keepdim=True))
        enh = self.pool(enh.mean(dim=1, keepdim=True))
        org_dh = org[:, :, :, 1:, :] - org[:, :, :, :-1, :]
        enh_dh = enh[:, :, :, 1:, :] - enh[:, :, :, :-1, :]
        org_dw = org[:, :, :, :, 1:] - org[:, :, :, :, :-1]
        enh_dw = enh[:, :, :, :, 1:] - enh[:, :, :, :, :-1]
        return (org_dh - enh_dh).pow(2).mean() + (org_dw - enh_dw).pow(2).mean()

class TVLoss3D(nn.Module):
    def forward(self, x):
        loss = 0.0
        if x.shape[2] > 1:
            loss = loss + (x[:, :, 1:, :, :] - x[:, :, :-1, :, :]).pow(2).mean()
        loss = loss + (x[:, :, :, 1:, :] - x[:, :, :, :-1, :]).pow(2).mean()
        loss = loss + (x[:, :, :, :, 1:] - x[:, :, :, :, :-1]).pow(2).mean()
        return loss

class EdgeLoss3D(nn.Module):
    def forward(self, org, enh):
        org = org.mean(dim=1, keepdim=True)
        enh = enh.mean(dim=1, keepdim=True)
        loss = 0.0
        if org.shape[2] > 1:
            loss = loss + torch.mean(torch.abs((org[:, :, 1:, :, :] - org[:, :, :-1, :, :]) - (enh[:, :, 1:, :, :] - enh[:, :, :-1, :, :])))
        loss = loss + torch.mean(torch.abs((org[:, :, :, 1:, :] - org[:, :, :, :-1, :]) - (enh[:, :, :, 1:, :] - enh[:, :, :, :-1, :])))
        loss = loss + torch.mean(torch.abs((org[:, :, :, :, 1:] - org[:, :, :, :, :-1]) - (enh[:, :, :, :, 1:] - enh[:, :, :, :, :-1])))
        return loss

class MSSSIMLossForClips(nn.Module):
    def forward(self, org, enh):
        b, c, d, h, w = org.shape
        org_2d = org.permute(0, 2, 1, 3, 4).reshape(b * d, c, h, w)
        enh_2d = enh.permute(0, 2, 1, 3, 4).reshape(b * d, c, h, w)
        if HAS_MSSSIM and min(h, w) >= 160:
            return 1.0 - ms_ssim(enh_2d, org_2d, data_range=1.0, size_average=True)
        return F.l1_loss(enh_2d, org_2d)

class Zero3DCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.color = ColorConstancyLoss3D()
        self.exp = ExposureLoss3D(patch_size=16, target=0.6)
        self.spa = SpatialConsistencyLoss3D(patch_size=4)
        self.tv = TVLoss3D()
        self.edge = EdgeLoss3D()
        self.msssim = MSSSIMLossForClips()
    def forward(self, low, enhanced, curves):
        losses = {}
        losses['exp'] = self.exp(enhanced)
        losses['tv'] = self.tv(curves)
        losses['spa'] = self.spa(low, enhanced)
        losses['edge'] = self.edge(low, enhanced)
        losses['color'] = self.color(enhanced)
        losses['msssim'] = self.msssim(low, enhanced)
        total = (10.0 * losses['exp'] + 200.0 * losses['tv'] + 1.0 * losses['spa'] + 2.0 * losses['edge'] + 5.0 * losses['color'] + 1.0 * losses['msssim'])
        losses['total'] = total
        return losses


In [10]:

# ===============================
# 6) TRAINING CONFIGURATION
# ===============================
class CFG:
    # Google Drive training frames
    train_root = TRAIN_FRAMES_DIR
    save_dir = SAVED_RUNS_DIR

    # Same practical setup used for thesis Colab training
    image_size = 518
    clip_len = 2
    stride = 1
    batch_size = 8

    # This trains for 50 total epochs, not 150.
    # It only saves selected checkpoints during that same 50-epoch run.
    epochs = 50
    save_epochs = {1, 10, 20, 30, 40, 50}

    lr = 1e-4
    weight_decay = 1e-4
    grad_clip = 0.1
    num_workers = 2

    # Use this only when continuing training from a previous checkpoint.
    # For a fresh run, keep it as None.
    resume_ckpt = None
    # Example:
    # resume_ckpt = SAVED_RUNS_DIR / 'zero3dce_best.pth'

Path(CFG.save_dir).mkdir(parents=True, exist_ok=True)
train_ds = LowLightClipDataset(CFG.train_root, clip_len=CFG.clip_len, image_size=CFG.image_size, stride=CFG.stride)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=True)
print('Training samples:', len(train_ds))
print('Example clip shape:', train_ds[0].shape)
print('Checkpoints will be saved to:', CFG.save_dir)


Training samples: 12242
Example clip shape: torch.Size([3, 2, 518, 518])
Checkpoints will be saved to: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs


In [11]:

# ===============================
# 7) TRAIN ZERO-3DCE WITH SELECTED + BEST CHECKPOINTS
# ===============================
model = Zero3DCE(nf=32).to(DEVICE)
model.apply(weights_init)
criterion = Zero3DCELoss().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)

history = []
best_loss = float('inf')
start_epoch = 1

# Optional resume logic
if CFG.resume_ckpt is not None and Path(CFG.resume_ckpt).exists():
    ckpt = torch.load(CFG.resume_ckpt, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    if 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    history = ckpt.get('history', [])
    best_loss = ckpt.get('best_loss', float('inf'))
    start_epoch = ckpt.get('epoch', 0) + 1
    print(f'Resumed from {CFG.resume_ckpt} at epoch {start_epoch}')
else:
    print('Starting fresh training run.')

for epoch in range(start_epoch, CFG.epochs + 1):
    model.train()
    running = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{CFG.epochs}')

    for clips in pbar:
        clips = clips.to(DEVICE, non_blocking=True)
        _, enhanced, curves = model(clips)
        losses = criterion(clips, enhanced, curves)
        loss = losses['total']

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        optimizer.step()

        running += loss.item()
        pbar.set_postfix({k: f'{v.item():.4f}' for k, v in losses.items()})

    avg = running / max(len(train_loader), 1)
    history.append(avg)
    print(f'Epoch {epoch}: avg_loss={avg:.4f}')

    checkpoint_payload = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'history': history,
        'avg_loss': avg,
        'best_loss': min(best_loss, avg),
        'config': {
            'image_size': CFG.image_size,
            'clip_len': CFG.clip_len,
            'batch_size': CFG.batch_size,
            'lr': CFG.lr,
            'train_root': str(CFG.train_root),
        }
    }

    # Save selected epoch checkpoints: 1, 10, 20, 30, 40, 50
    if epoch in CFG.save_epochs:
        ckpt_path = Path(CFG.save_dir) / f'zero3dce_epoch_{epoch}.pth'
        torch.save(checkpoint_payload, ckpt_path)
        print('Saved epoch checkpoint:', ckpt_path)

    # Save best checkpoint by lowest training loss
    if avg < best_loss:
        best_loss = avg
        checkpoint_payload['best_loss'] = best_loss
        best_path = Path(CFG.save_dir) / 'zero3dce_best.pth'
        torch.save(checkpoint_payload, best_path)
        print('Saved new best checkpoint:', best_path, '| best_loss:', best_loss)

print('Training finished.')
print('Best checkpoint:', Path(CFG.save_dir) / 'zero3dce_best.pth')
print('Selected checkpoints saved at:', sorted(CFG.save_epochs))


Starting fresh training run.


Epoch 1/50:   0%|          | 0/1531 [00:00<?, ?it/s]

Epoch 1: avg_loss=1.5464
Saved epoch checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_epoch_1.pth
Saved new best checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_best.pth | best_loss: 1.546364319783889


Epoch 2/50:   0%|          | 0/1531 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7803e6b12d40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7803e6b12d40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

...

Epoch 48: avg_loss=0.7572
Saved new best checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_best.pth | best_loss: 0.7572274529451175


Epoch 49/50:   0%|          | 0/1531 [00:00<?, ?it/s]

Epoch 49: avg_loss=0.7567
Saved new best checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_best.pth | best_loss: 0.7566647547900015


Epoch 50/50:   0%|          | 0/1531 [00:00<?, ?it/s]

Epoch 50: avg_loss=0.7563
Saved epoch checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_epoch_50.pth
Saved new best checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_best.pth | best_loss: 0.756333534815668
Training finished.
Best checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_best.pth
Selected checkpoints saved at: [1, 10, 20, 30, 40, 50]


In [12]:

# ===============================
# 8) INFERENCE ON IMAGE OR VIDEO FRAMES
# ===============================
@torch.no_grad()
def enhance_image_file(model, image_path, out_path, image_size=518, clip_len=2):
    model.eval()
    tf = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])
    img = Image.open(image_path).convert('RGB')
    x = tf(img).unsqueeze(0)
    clip = x.unsqueeze(2).repeat(1, 1, clip_len, 1, 1).to(DEVICE)
    _, enhanced, _ = model(clip)
    y = enhanced[:, :, clip_len // 2, :, :]
    save_image(y, out_path)
    return out_path

@torch.no_grad()
def enhance_frame_folder(model, in_dir, out_dir, image_size=518, clip_len=2):
    model.eval()
    in_dir, out_dir = Path(in_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    frames = sorted([p for p in in_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    tf = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])
    if len(frames) == 0:
        raise RuntimeError('No frames found.')
    padded = [frames[0]] * (clip_len // 2) + frames + [frames[-1]] * (clip_len // 2)
    for i in tqdm(range(len(frames)), desc='Enhancing frames'):
        clip_paths = padded[i:i+clip_len]
        if len(clip_paths) < clip_len:
            clip_paths = clip_paths + [clip_paths[-1]] * (clip_len - len(clip_paths))
        imgs = [tf(Image.open(p).convert('RGB')) for p in clip_paths]
        clip = torch.stack(imgs, dim=1).unsqueeze(0).to(DEVICE)
        _, enhanced, _ = model(clip)
        y = enhanced[:, :, min(clip_len // 2, enhanced.shape[2]-1), :, :]
        save_image(y, out_dir / frames[i].name)
    print('Saved enhanced frames to:', out_dir)

# Load the best checkpoint before testing/enhancement.
best_path = Path(CFG.save_dir) / 'zero3dce_best.pth'
if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    print('Loaded best checkpoint:', best_path)
else:
    print('No best checkpoint found yet. Using the current model in memory.')

# Enhance the extracted evening test frames.
enhance_frame_folder(model, TEST_FRAMES_DIR, ENHANCED_FRAMES_DIR, CFG.image_size, CFG.clip_len)


Loaded best checkpoint: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/saved_runs/zero3dce_best.pth


Enhancing frames:   0%|          | 0/12242 [00:00<?, ?it/s]

Saved enhanced frames to: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/outputs_enhanced_frames/evening_enhanced_frames


In [14]:

# ===============================
# 9) REBUILD ENHANCED FRAMES INTO VIDEO
# ===============================
def frames_to_video(frame_dir, out_video, fps=6):
    frame_dir = Path(frame_dir)
    out_video = Path(out_video)
    out_video.parent.mkdir(parents=True, exist_ok=True)
    frames = sorted([p for p in frame_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    if not frames:
        raise RuntimeError('No frames found.')
    first = cv2.imread(str(frames[0]))
    h, w = first.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(out_video), fourcc, fps, (w, h))
    for p in frames:
        img = cv2.imread(str(p))
        if img.shape[:2] != (h, w):
            img = cv2.resize(img, (w, h))
        writer.write(img)
    writer.release()
    print('Saved video:', out_video)

frames_to_video(ENHANCED_FRAMES_DIR, ENHANCED_VIDEO_PATH, fps=6)


Saved video: /content/drive/MyDrive/Thesis2/outputs_zero3dce_v4sofia/outputs_enhanced_videos/evening_zero3dce_enhanced.mp4


## Notes

- `zero3dce_best.pth` is saved using the lowest training loss in this notebook. This is useful, but it is not the same as a validation-based best checkpoint.
- The selected checkpoints `zero3dce_epoch_1.pth`, `zero3dce_epoch_10.pth`, `zero3dce_epoch_20.pth`, `zero3dce_epoch_30.pth`, `zero3dce_epoch_40.pth`, and `zero3dce_epoch_50.pth` are saved so you can visually compare outputs per epoch.
- `epochs = 50` means the training loop runs for 50 total epochs. It does not run 1 + 10 + 20 + 30 + 40 + 50 epochs separately.
